# RAG 시스템 평가 노트북

**대상:** 포항시 시설관리 RAG (Chroma + ko-sroberta + gemma4:e4b)

## 평가 축
1. **검색 품질** — hit@k, MRR, precision@k (골든셋 기반)
2. **응답 시간** — retrieval 및 end-to-end latency
3. **견적 정확도** — MAPE (est DB leave-one-out)
4. **챗봇 피드백** — 실서비스 `chat_logs.json` 의 👍/👎 집계
5. **(선택) Ragas** — faithfulness, answer_relevancy (Gemini API Key 필요)

## 실행 전제
- `server/` 에서 import 가능해야 함. 서버 프로세스 자체는 안 돌아도 됨(서비스 직접 호출).
- Chroma 인덱스가 빌드돼 있어야 함 (`chroma_est/`).

## 1. 환경 설정

In [1]:
import sys, os, json, time, statistics
from pathlib import Path
from collections import Counter, defaultdict

# server/ 를 import path 에 추가 (노트북 위치가 server/scripts/ 인 가정)
SERVER = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(SERVER))
print('SERVER dir:', SERVER)

from services import storage, rag
print('Chroma collection:', storage.collection().count(), '개 문서')

SERVER dir: /home/piai/다운로드/llm및 data/server


/home/piai/anaconda3/envs/aienv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma collection: 21926 개 문서


## 2. 골든셋 정의

**각 항목:**
- `query`: 사용자 질문 그대로
- `expected_item`: (검색 결과 metadata의) Item 필드에 이게 포함되면 hit
- `expected_source`: 'est' | 'rep' | 'emg' | 'live' | None (무관)
- `min_damage`: 있으면 해당 값 이상의 손상도만 hit 로 인정 (견적용)
- `keywords`: document 텍스트에 반드시 포함돼야 할 단어 (옵션)

**아래 골든셋은 템플릿. 프로젝트 domain에 맞게 30~50개로 늘려주세요.**

In [2]:
GOLDEN = [
  # --- 견적 검색 ---
  {'query': '보호펜스 파손 3단계 수리비', 'expected_item': '보호펜스', 'expected_source': 'est', 'min_damage': 3},
  {'query': '가로등 완파 교체 비용', 'expected_item': '가로등', 'expected_source': 'est'},
  {'query': '도로 구멍 보수 견적', 'expected_item': '도로', 'expected_source': 'est'},
  {'query': '볼라드 충돌 손상 수리', 'expected_item': '볼라드', 'expected_source': 'est', 'min_damage': 2},
  {'query': '맨홀 파손 교체 비용', 'expected_item': '맨홀', 'expected_source': 'est'},

  # --- 신고 이력 검색 ---
  {'query': '보호펜스 최근 신고', 'expected_item': '보호펜스', 'expected_source': 'rep'},
  {'query': '등받이 벤치 파손 접수', 'expected_item': '등받이', 'expected_source': 'rep'},
  {'query': '미끄럼틀 손상 신고', 'expected_item': '미끄럼틀', 'expected_source': 'rep'},

  # --- 긴급도 ---
  {'query': '학교 근처 가로등 위험 긴급', 'expected_source': 'emg', 'keywords': ['학교']},
  {'query': '보행자 위험 시설물', 'expected_source': 'emg'},

  # --- Live 신고 ---
  {'query': '접수된 신고 중 보호펜스', 'expected_item': '보호펜스', 'expected_source': 'live'},
]
print(f'골든셋 {len(GOLDEN)}개')

골든셋 11개


## 3. 검색(Retrieval) 평가 — hit@k, MRR, precision@k

**판정 기준**
- `expected_item` 이 metadata.Item 에 포함되고, `expected_source` 가 source 와 일치하면 hit
- `min_damage` 있으면 Damage_Rate ≥ min_damage 도 추가 조건
- `keywords` 있으면 document 텍스트에 모두 포함돼야 함

In [3]:
def is_relevant(hit, g):
    m = hit['metadata'] or {}
    doc = hit.get('document', '') or ''
    if g.get('expected_source') and m.get('source') != g['expected_source']:
        return False
    if g.get('expected_item') and g['expected_item'] not in (m.get('Item') or ''):
        return False
    if g.get('min_damage') is not None:
        try:
            if int(m.get('Damage_Rate', 0)) < int(g['min_damage']):
                return False
        except Exception:
            return False
    for kw in g.get('keywords', []):
        if kw not in doc:
            return False
    return True

def eval_retrieval(golden, k=8):
    rows = []
    for g in golden:
        t0 = time.time()
        hits = rag.search(g['query'], k=k, source=g.get('expected_source'))
        dt = (time.time() - t0) * 1000
        # hit position (1-based, 0=miss)
        rank = 0
        rel_count = 0
        for i, h in enumerate(hits, 1):
            if is_relevant(h, g):
                rel_count += 1
                if rank == 0:
                    rank = i
        rows.append({
            'query': g['query'][:40],
            'top_sim': hits[0]['similarity'] if hits else 0,
            'rank_first_hit': rank or '-',
            'relevant_in_topk': rel_count,
            'latency_ms': round(dt, 1),
            'hit@5': 1 if 0 < rank <= 5 else 0,
            'hit@8': 1 if 0 < rank <= 8 else 0,
            'rr': 1.0/rank if rank else 0,
            'p@5': rel_count / min(5, len(hits)) if hits else 0,
        })
    return rows

rows = eval_retrieval(GOLDEN, k=8)
import pandas as pd
df = pd.DataFrame(rows)
print(f"\n=== 집계 ===")
print(f"Hit@5        : {df['hit@5'].mean():.2%}")
print(f"Hit@8        : {df['hit@8'].mean():.2%}")
print(f"MRR          : {df['rr'].mean():.3f}")
print(f"Precision@5  : {df['p@5'].mean():.2%}")
print(f"평균 latency : {df['latency_ms'].mean():.0f} ms")
print(f"miss (rank=-): {(df['rank_first_hit']=='-').sum()} / {len(df)}")
df.head(20)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



=== 집계 ===
Hit@5        : 63.64%
Hit@8        : 63.64%
MRR          : 0.636
Precision@5  : 100.00%
평균 latency : 579 ms
miss (rank=-): 4 / 11


,query,top_sim,rank_first_hit,relevant_in_topk,latency_ms,hit@5,hit@8,rr,p@5
0,보호펜스 파손 3단계 수리비,0.749,1,8,4933.9,1,1,1.0,1.6
1,가로등 완파 교체 비용,0.482,-,0,356.1,0,0,0.0,0.0
2,도로 구멍 보수 견적,0.618,-,0,150.4,0,0,0.0,0.0
3,볼라드 충돌 손상 수리,0.479,1,7,181.2,1,1,1.0,1.4
4,맨홀 파손 교체 비용,0.615,1,8,145.3,1,1,1.0,1.6
5,보호펜스 최근 신고,0.555,1,8,160.5,1,1,1.0,1.6
6,등받이 벤치 파손 접수,0.570,1,8,145.0,1,1,1.0,1.6
7,미끄럼틀 손상 신고,0.595,1,8,147.5,1,1,1.0,1.6
8,학교 근처 가로등 위험 긴급,0.348,-,0,43.7,0,0,0.0,0.0
9,보행자 위험 시설물,0.341,1,8,61.7,1,1,1.0,1.6


### 3-1. 실패(miss) 케이스 조사

어떤 쿼리에서 miss 가 나는지 확인 → 골든셋 보정 or 임베딩·검색 개선.

In [4]:
misses = df[df['rank_first_hit']=='-']
print(f'miss {len(misses)}건\n')
for _, r in misses.iterrows():
    print(f"❌ '{r['query']}'")
    g = next((g for g in GOLDEN if g['query'].startswith(r['query'][:30])), None)
    if g:
        hits = rag.search(g['query'], k=3, source=g.get('expected_source'))
        for h in hits:
            m = h['metadata'] or {}
            print(f"   top : sim={h['similarity']} src={m.get('source')} Item={m.get('Item')}")
    print()

miss 4건

❌ '가로등 완파 교체 비용'
   top : sim=0.482 src=est Item=가로수보호덮개
   top : sim=0.482 src=est Item=가로수보호덮개
   top : sim=0.482 src=est Item=가로수보호덮개

❌ '도로 구멍 보수 견적'
   top : sim=0.618 src=est Item=보도블록
   top : sim=0.601 src=est Item=보도블록
   top : sim=0.597 src=est Item=보도블록

❌ '학교 근처 가로등 위험 긴급'
   top : sim=0.348 src=emg Item=None
   top : sim=0.339 src=emg Item=None
   top : sim=0.339 src=emg Item=None

❌ '접수된 신고 중 보호펜스'
   top : sim=0.419 src=live Item=등받이없는벤치
   top : sim=0.396 src=live Item=가로수보호덮개
   top : sim=0.372 src=live Item=농구대



## 4. 견적 정확도 — MAPE (est DB leave-one-out)

est 소스에서 N개 샘플링 → 각 건의 `Item` + `Damage_Rate` 로 쿼리 생성 →
예측 중앙값(stats.median) vs 실제 Cost 비교. **Mean Absolute Percentage Error** 출력.

> 주의: leave-one-out 완전 적용하려면 해당 건 제외 후 재쿼리해야 함. 여기선 간이(자기참조 허용) 버전. 필요 시 `filter_self=True` 로 strict 모드.

In [5]:
import random
random.seed(42)

col = storage.collection()
got = col.get(where={'source': 'est'}, limit=20000)
all_items = list(zip(got['ids'], got['documents'], got['metadatas']))
sample = random.sample(all_items, min(50, len(all_items)))

errors = []
for doc_id, doc, meta in sample:
    item = meta.get('Item', '')
    dr   = meta.get('Damage_Rate', '')
    actual = float(meta.get('Cost', 0))
    if actual <= 0:
        continue
    query = f'{item} 손상도 {dr}단계 수리 견적'
    res = rag.estimate(query, k=8, item=item, damage_rate=int(dr) if dr else None)
    pred = (res.get('stats') or {}).get('median')
    if pred is None:
        continue
    ape = abs(pred - actual) / actual * 100
    errors.append({'id': doc_id, 'item': item, 'damage': dr,
                   'actual': int(actual), 'pred_median': int(pred), 'ape_%': round(ape, 1)})
df_mape = pd.DataFrame(errors).sort_values('ape_%', ascending=False)
print(f'샘플 {len(df_mape)}건')
print(f'MAPE      : {df_mape["ape_%"].mean():.2f}%')
print(f'median APE: {df_mape["ape_%"].median():.2f}%')
print(f'APE 상위 10:')
df_mape.head(10)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


샘플 50건
MAPE      : 18.31%
median APE: 13.75%
APE 상위 10:


,id,item,damage,actual,pred_median,ape_%
36,est_EST_01520,미끄럼틀,1,687000,1177500,71.4
32,est_EST_02548,보도블록,1,7000,12000,71.4
3,est_EST_04013,축구골대,2,1396000,2131500,52.7
37,est_EST_06225,맨홀,2,425000,616500,45.1
28,est_EST_02616,등받이있는벤치,2,278000,403000,45.0
8,est_EST_01425,등받이없는벤치,1,142000,85000,40.1
14,est_EST_03583,맨홀,3,831000,1096500,31.9
25,est_EST_09655,미끄럼틀,3,5511000,3893500,29.4
12,est_EST_00489,축구골대,1,831000,1065000,28.2
10,est_EST_06913,등받이없는벤치,1,118000,85000,28.0


## 5. 실서비스 챗봇 피드백 집계

`chat_logs.json` 의 👍/👎 추이. 사용자 피드백이 누적될수록 가치 있음.

In [ ]:
chat_log_path = SERVER / 'chat_logs.json'
if not chat_log_path.exists():
    print('chat_logs.json 없음 (민원인 챗봇 이용 후 생김)')
else:
    logs = json.loads(chat_log_path.read_text())
    total = len(logs)
    good = sum(1 for r in logs if r.get('feedback') == 'good')
    bad  = sum(1 for r in logs if r.get('feedback') == 'bad')
    none = total - good - bad
    print(f'대화 턴 총 {total}건 — 👍 {good} · 👎 {bad} · 미평가 {none}')
    print(f'평가률 : {(good+bad)/total:.1%}' if total else '')
    print(f'만족도 : {good/(good+bad):.1%}' if (good+bad) else '피드백 부족')

    # 👎 사유 출력
    print('\n=== 👎 사례 ===')
    for r in logs:
        if r.get('feedback') == 'bad':
            print(f"  [{r['timestamp'][:16]}] {r['user_name']} — Q: {r['message'][:60]}")
            if r.get('feedback_note'):
                print(f'     note: {r["feedback_note"][:100]}')

## 6. (선택) Ragas 자동 평가 — faithfulness / answer_relevancy

Gemini API 필요. `export GOOGLE_API_KEY=xxx` 후 셀 실행. 아니면 스킵.

In [ ]:
RAGAS_ENABLED = bool(os.getenv('GOOGLE_API_KEY'))
print('Ragas enabled:', RAGAS_ENABLED)

if RAGAS_ENABLED:
    try:
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision
        from ragas.llms import LangchainLLMWrapper
        from langchain_google_genai import ChatGoogleGenerativeAI
        from datasets import Dataset

        # 평가용 샘플: 골든셋에서 LLM 답변 뽑아 평가
        eval_rows = []
        for g in GOLDEN[:5]:   # 5개만 샘플 (토큰 비용 주의)
            res = rag.estimate(g['query'], k=6,
                                item=g.get('expected_item'),
                                damage_rate=g.get('min_damage'))
            eval_rows.append({
                'question': g['query'],
                'answer':   res.get('reply', ''),
                'contexts': res.get('cases', []),
            })
        ds = Dataset.from_list(eval_rows)
        llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model='gemini-2.0-flash'))
        report = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision], llm=llm)
        print(report)
    except Exception as e:
        print('Ragas 평가 실패:', e)
else:
    print('GOOGLE_API_KEY 미설정 — Ragas 스킵')

## 7. 스냅샷 저장 — 일자별 비교용

In [ ]:
from datetime import date

report = {
    'date': date.today().isoformat(),
    'retrieval': {
        'n_queries': len(GOLDEN),
        'hit@5':     round(df['hit@5'].mean(), 4),
        'hit@8':     round(df['hit@8'].mean(), 4),
        'mrr':       round(df['rr'].mean(), 4),
        'precision@5': round(df['p@5'].mean(), 4),
        'avg_latency_ms': round(df['latency_ms'].mean(), 1),
        'miss_count': int((df['rank_first_hit']=='-').sum()),
    },
    'estimate': {
        'n_samples': len(df_mape),
        'mape_percent': round(df_mape['ape_%'].mean(), 2),
        'median_ape_percent': round(df_mape['ape_%'].median(), 2),
    },
}

out_dir = SERVER / 'eval_runs'
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"{report['date']}.json"
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f'저장: {out_path}')
print(json.dumps(report, ensure_ascii=False, indent=2))

## 사용 팁

- **골든셋 규모**: 최소 30건 이상 권장. 50~100건이면 안정적.
- **실패 분석**: 셀 3-1의 miss 리스트를 보고 검색어 대체·임베딩 교체·filter 조정 가이드.
- **정기 실행**: 배포 직전 매번 돌려서 `eval_runs/YYYY-MM-DD.json` 으로 비교.
- **골든셋 관리**: 이 노트북 밖에 `eval_runs/golden_set.json` 로 분리해도 좋음.